# Label Scenario 4 

In [2]:
#
# Cell 1: Libraries laden
#

import pandas as pd
import numpy as np
import wfdb
import ast

In [11]:
#
# Cell 2:Path for input data and definition of sampling rate
#

#path = 'C:/Users/Buchh/OneDrive/Leuphana_Data_Science/Masterarbeit/ECG/Python/Daten_einlesen/test_data/'
path = 'E:/MA_Daten_2/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/'
sampling_rate=100


In [13]:
#
# Cell 3: Definition of functions
#

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

def aggregate_form_rhythm(y_dic): #BuK
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].form_rhythm)
    return list(set(tmp))


In [15]:
#
# Cell 3a: Specify the directory name for loading of data
#

from pathlib import Path

directory_path = Path("D:/ECG/X_zero")

In [17]:
#
# Cell 4:Loading stored raw data, if available. 
#

#X = np.load('X_np_array.npy')
Y = pd.read_pickle(directory_path/'Y_data.pkl')



In [19]:
#
# Cell 5: FORM AND RHYTHM LABELS
# Load scp_statements.csv for form and rhythm aggregation (BuK)

agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[(agg_df.form == 1) | (agg_df.rhythm == 1)]
agg_df['form_rhythm'] = agg_df.index



In [21]:
#
#Cell 6: Apply form and rhythm class, BuK
#

Y['label'] = Y.scp_codes.apply(aggregate_form_rhythm)
display(Y)



,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,label
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,"[SR, LVOLT]"
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,[SBRAD]
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,[SR]
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,[SR]
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,[SR]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21794,21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,...,NaN,", alles,",NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr,"[VCLVH, NDT, STACH, PVC]"
21795,21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,...,NaN,NaN,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr,"[ABQRS, SR]"
21796,21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,...,NaN,", I-AVR,",NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr,[SR]
21797,21836,8873.0,64.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-09 18:21:49,...,NaN,NaN,NaN,NaN,SVES,NaN,8,records100/21000/21836_lr,records500/21000/21836_hr,[SR]


In [23]:
#
#Cell 7: Make a copy from Y for further manipulation
#

Y_copy = Y[['ecg_id','patient_id','strat_fold','label']].copy() 
Y_copy

,ecg_id,patient_id,strat_fold,label
0,1,15709.0,3,"[SR, LVOLT]"
1,2,13243.0,2,[SBRAD]
2,3,20372.0,5,[SR]
3,4,17014.0,3,[SR]
4,5,17448.0,4,[SR]
...,...,...,...,...
21794,21833,17180.0,7,"[VCLVH, NDT, STACH, PVC]"
21795,21834,20703.0,4,"[ABQRS, SR]"
21796,21835,19311.0,2,[SR]
21797,21836,8873.0,8,[SR]


## Y-data manipulation for augmentation starts 

In [18]:
#
# Cell 7a: 
#

from pathlib import Path
directory_path_augm = Path("d:/ECG/X_augmented")

In [20]:
# Cell 7b: loads dicitionary with augmented X-xata

import pickle

with open(directory_path_augm/'X_augmented_accomp.pkl', 'rb') as f:
    X_augmented_accomp = pickle.load(f) # deserialize using load()
f.close()  

X_augmented_accomp

{'# of additional X-Data concatenated': 2}

In [22]:
# Cell 7c: select a certain row for later control purposes:
Y_21794 = Y_copy[Y_copy.index==21794].copy()
Y_21794

,ecg_id,patient_id,strat_fold,label
21794,21833,17180.0,7,"[NDT, STACH, PVC, VCLVH]"


In [24]:
# Cell 7d: concatenates the df Y_copy according to the value of  
# '# of additional X-Data concatenated':
Y_add=Y_copy
for i in range(0, X_augmented_accomp.get('# of additional X-Data concatenated')):
    Y_copy= pd.concat([Y_copy,Y_add], axis=0, ignore_index= True).copy()

Y_copy    

,ecg_id,patient_id,strat_fold,label
0,1,15709.0,3,"[LVOLT, SR]"
1,2,13243.0,2,[SBRAD]
2,3,20372.0,5,[SR]
3,4,17014.0,3,[SR]
4,5,17448.0,4,[SR]
...,...,...,...,...
65392,21833,17180.0,7,"[NDT, STACH, PVC, VCLVH]"
65393,21834,20703.0,4,"[ABQRS, SR]"
65394,21835,19311.0,2,[SR]
65395,21836,8873.0,8,[SR]


In [26]:
# Cell 7e: compare two df extracts for control of correct index assignment  
# 21749 + 21799 = 43593

Y_43593 = Y_copy[Y_copy.index==43593].copy()
display(Y_43593)
display(Y_21794)

,ecg_id,patient_id,strat_fold,label
43593,21833,17180.0,7,"[NDT, STACH, PVC, VCLVH]"


,ecg_id,patient_id,strat_fold,label
21794,21833,17180.0,7,"[NDT, STACH, PVC, VCLVH]"


## Y-data manipulation for augmentation ends
## The remaining cells are indentical with the cells of the modul for not augmented data   

In [25]:
#
# Cell 8: in all combinations of labels with "SR" remove "SR" 
# clean list of labels is now stored in the column "codes"
#

Y_copy["codes"] = Y_copy["label"].apply(
    lambda lst: ["SR"] if len(lst) == 1 and lst[0] == "SR"
    else sorted([c for c in lst if c != "SR"])
)
Y_copy

,ecg_id,patient_id,strat_fold,label,codes
0,1,15709.0,3,"[SR, LVOLT]",[LVOLT]
1,2,13243.0,2,[SBRAD],[SBRAD]
2,3,20372.0,5,[SR],[SR]
3,4,17014.0,3,[SR],[SR]
4,5,17448.0,4,[SR],[SR]
...,...,...,...,...,...
21794,21833,17180.0,7,"[VCLVH, NDT, STACH, PVC]","[NDT, PVC, STACH, VCLVH]"
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS]
21796,21835,19311.0,2,[SR],[SR]
21797,21836,8873.0,8,[SR],[SR]


In [27]:
#
# Cell 9: count the items in the lists in column "codes" and write to "len" column 
#

Y_copy["len"] = Y_copy["codes"].apply(
    lambda lst: len(lst)
)
Y_copy


,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
21794,21833,17180.0,7,"[VCLVH, NDT, STACH, PVC]","[NDT, PVC, STACH, VCLVH]",4
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
21796,21835,19311.0,2,[SR],[SR],1
21797,21836,8873.0,8,[SR],[SR],1


Cell 10 to cell 17: all ECGs with one label
Cell ... to cel ...: all ECGs with two label

In [29]:
#
# Cell 10. filter all instaces with one label
#

Y_label_len_one_df = Y_copy[Y_copy["len"]==1].copy()
Y_label_len_one_df

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",[LVOLT],1
1,2,13243.0,2,[SBRAD],[SBRAD],1
2,3,20372.0,5,[SR],[SR],1
3,4,17014.0,3,[SR],[SR],1
4,5,17448.0,4,[SR],[SR],1
...,...,...,...,...,...,...
21793,21832,7954.0,7,[SR],[SR],1
21795,21834,20703.0,4,"[ABQRS, SR]",[ABQRS],1
21796,21835,19311.0,2,[SR],[SR],1
21797,21836,8873.0,8,[SR],[SR],1


In [31]:
#
# Cell 11:ONE LABEL - convert list element of column "codes" into objects
#

Y_label_len_one_df['codes'] = Y_label_len_one_df['codes'].apply(lambda x: "".join(x)).copy()
Y_label_len_one_df

,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
...,...,...,...,...,...,...
21793,21832,7954.0,7,[SR],SR,1
21795,21834,20703.0,4,"[ABQRS, SR]",ABQRS,1
21796,21835,19311.0,2,[SR],SR,1
21797,21836,8873.0,8,[SR],SR,1


In [33]:
#
# Cell 11a: only information
#

Y_label_len_one_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17504 entries, 0 to 21798
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ecg_id      17504 non-null  int64  
 1   patient_id  17504 non-null  float64
 2   strat_fold  17504 non-null  int64  
 3   label       17504 non-null  object 
 4   codes       17504 non-null  object 
 5   len         17504 non-null  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 957.2+ KB


In [35]:
#
# Cell 12: ONE LABEL - count number of each code in column "codes" 
#

combo_counts = Y_label_len_one_df["codes"].value_counts()
combo_counts

#combo_counts.to_csv("combo_counts.csv")

codes
SR        10125
ABQRS      2332
NDT         862
AFIB        481
SARRH       472
SBRAD       428
VCLVH       399
STACH       370
PVC         316
PACE        281
NST_        271
QWAVE       224
STD_        142
LPR         136
PAC         127
NT_         101
LVOLT        88
SVARR        74
LNGQT        62
INVT         52
HVOLT        43
AFLT         28
LOWT         27
DIG          13
TAB_         12
STE_         11
BIGU         10
PSVT          8
SVTAC         5
PRC(S)        3
TRIGU         1
Name: count, dtype: int64

In [37]:
#
# Cell 12a: information 
#

type(combo_counts)

pandas.core.series.Series

In [39]:
#
# Cell 13: ONE LABEL - coverts series into df 
#

combo_counts_df = combo_counts.to_frame()
combo_counts_df

,count
codes,
SR,10125
ABQRS,2332
NDT,862
AFIB,481
SARRH,472
SBRAD,428
VCLVH,399
STACH,370
PVC,316


In [41]:
#
# Cell 13a: information
#

combo_counts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31 entries, SR to TRIGU
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   count   31 non-null     int64
dtypes: int64(1)
memory usage: 1.5+ KB


In [43]:
#
# Cell 14: ONE LABEL - restrict codes to n.l.t. 49 instances  
#

combo_count_short_df=combo_counts_df[combo_counts_df['count']>=49]
combo_count_short_df

,count
codes,
SR,10125
ABQRS,2332
NDT,862
AFIB,481
SARRH,472
SBRAD,428
VCLVH,399
STACH,370
PVC,316


In [45]:
#
# Cell 15: ONE LABEL - reorders df  
#

combo_count_short_reset_df = combo_count_short_df.reset_index( ) 
combo_count_short_reset_df

,codes,count
0,SR,10125
1,ABQRS,2332
2,NDT,862
3,AFIB,481
4,SARRH,472
5,SBRAD,428
6,VCLVH,399
7,STACH,370
8,PVC,316
9,PACE,281


In [47]:
#
# Cell 15a: information  
#

combo_count_short_reset_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   codes   20 non-null     object
 1   count   20 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 452.0+ bytes


In [49]:
#
# Cell 16: ONE LABEL - convert column "codes" to list  
#

codes_list =combo_count_short_reset_df['codes'].to_list() 
codes_list
#df = df[df.index.isin(df1.index)]

['SR',
 'ABQRS',
 'NDT',
 'AFIB',
 'SARRH',
 'SBRAD',
 'VCLVH',
 'STACH',
 'PVC',
 'PACE',
 'NST_',
 'QWAVE',
 'STD_',
 'LPR',
 'PAC',
 'NT_',
 'LVOLT',
 'SVARR',
 'LNGQT',
 'INVT']

In [51]:
#
# Cell 17: ONE LABEL - filters from df "Y_label_len_one_df" all rows with codes in column "codes" 
# that are contained in the list codeslistconvert column "codes" to list
#

Y_label_len_one_short_df= Y_label_len_one_df[Y_label_len_one_df['codes'].isin(codes_list)]
Y_label_len_one_short_df


,ecg_id,patient_id,strat_fold,label,codes,len
0,1,15709.0,3,"[SR, LVOLT]",LVOLT,1
1,2,13243.0,2,[SBRAD],SBRAD,1
2,3,20372.0,5,[SR],SR,1
3,4,17014.0,3,[SR],SR,1
4,5,17448.0,4,[SR],SR,1
...,...,...,...,...,...,...
21793,21832,7954.0,7,[SR],SR,1
21795,21834,20703.0,4,"[ABQRS, SR]",ABQRS,1
21796,21835,19311.0,2,[SR],SR,1
21797,21836,8873.0,8,[SR],SR,1


Cell ... to cell ...: combine "Y_label_len_two_short_df" and "Y_label_len_two_short_df"
and substract items from the ... to get the remianing EGS, which are not contained in the two classes

In [53]:
#
# Cell 18: ALL OTHER - substract Y_label_len_one_short_df from Y_copy
#

Y_all_other = Y_copy[~Y_copy.index.isin(Y_label_len_one_short_df.index)]
Y_all_other


,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[ABQRS, AFIB, AFLT]","[ABQRS, AFIB, AFLT]",3
17,18,13619.0,9,[AFLT],[AFLT],1
19,20,13619.0,9,"[ABQRS, AFLT]","[ABQRS, AFLT]",2
22,23,13619.0,9,[AFLT],[AFLT],1
25,26,13619.0,9,"[DIG, NST_, SR]","[DIG, NST_]",2
...,...,...,...,...,...,...
21779,21818,19204.0,10,"[ABQRS, SR, VCLVH]","[ABQRS, VCLVH]",2
21787,21826,9178.0,10,"[ABQRS, SARRH]","[ABQRS, SARRH]",2
21788,21827,13862.0,5,"[ABQRS, SVARR]","[ABQRS, SVARR]",2
21789,21828,13862.0,5,"[ABQRS, SR, PVC]","[ABQRS, PVC]",2


In [55]:
#
# Cell 19: ALL OTHER - filter from Y_all_other all instances with a empty label
#

Y_label_len_zero_df = Y_all_other[Y_all_other["len"]==0].copy()
Y_label_len_zero_df

,ecg_id,patient_id,strat_fold,label,codes,len
134,135,19453.0,6,[],[],0
143,151,12152.0,9,[],[],0
156,164,15095.0,3,[],[],0
173,181,21551.0,4,[],[],0
178,186,12396.0,9,[],[],0
...,...,...,...,...,...,...
21501,21540,21602.0,2,[],[],0
21551,21590,12310.0,6,[],[],0
21591,21630,10139.0,6,[],[],0
21593,21632,13743.0,1,[],[],0


In [57]:
#
# Cell 20: ALL OTHER - filter from Y_all_other all instances with a empty label
#

Y_all_other_pure = Y_all_other[~Y_all_other.index.isin(Y_label_len_zero_df.index)]
Y_all_other_pure

,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[ABQRS, AFIB, AFLT]","[ABQRS, AFIB, AFLT]",3
17,18,13619.0,9,[AFLT],[AFLT],1
19,20,13619.0,9,"[ABQRS, AFLT]","[ABQRS, AFLT]",2
22,23,13619.0,9,[AFLT],[AFLT],1
25,26,13619.0,9,"[DIG, NST_, SR]","[DIG, NST_]",2
...,...,...,...,...,...,...
21779,21818,19204.0,10,"[ABQRS, SR, VCLVH]","[ABQRS, VCLVH]",2
21787,21826,9178.0,10,"[ABQRS, SARRH]","[ABQRS, SARRH]",2
21788,21827,13862.0,5,"[ABQRS, SVARR]","[ABQRS, SVARR]",2
21789,21828,13862.0,5,"[ABQRS, SR, PVC]","[ABQRS, PVC]",2


In [59]:
#
# Cell 21:  assign a unique code to all ECGs of one class (form, rhytm, all other)    
#

df_list= [[Y_all_other_pure, 'all_other']]

df_list
for item in df_list:
    #item[0]['codes_recover'] = item[0]['codes']
    item[0].loc[:,'codes'] = item[1]
    

Y_all_other_pure

,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[ABQRS, AFIB, AFLT]",all_other,3
17,18,13619.0,9,[AFLT],all_other,1
19,20,13619.0,9,"[ABQRS, AFLT]",all_other,2
22,23,13619.0,9,[AFLT],all_other,1
25,26,13619.0,9,"[DIG, NST_, SR]",all_other,2
...,...,...,...,...,...,...
21779,21818,19204.0,10,"[ABQRS, SR, VCLVH]",all_other,2
21787,21826,9178.0,10,"[ABQRS, SARRH]",all_other,2
21788,21827,13862.0,5,"[ABQRS, SVARR]",all_other,2
21789,21828,13862.0,5,"[ABQRS, SR, PVC]",all_other,2


In [61]:
#
# Cell 22: makes a dictionary from list codes_list
#

dfs= {}
for item in codes_list:
    #dfs['Y_'+item] = item
    dfs[item] = item
dfs

{'SR': 'SR',
 'ABQRS': 'ABQRS',
 'NDT': 'NDT',
 'AFIB': 'AFIB',
 'SARRH': 'SARRH',
 'SBRAD': 'SBRAD',
 'VCLVH': 'VCLVH',
 'STACH': 'STACH',
 'PVC': 'PVC',
 'PACE': 'PACE',
 'NST_': 'NST_',
 'QWAVE': 'QWAVE',
 'STD_': 'STD_',
 'LPR': 'LPR',
 'PAC': 'PAC',
 'NT_': 'NT_',
 'LVOLT': 'LVOLT',
 'SVARR': 'SVARR',
 'LNGQT': 'LNGQT',
 'INVT': 'INVT'}

In [63]:
#
# Cell 23: splitts the set of all single labels with instances g.t. 49 into indiviual sets
#


individual_dfs = {
    name: Y_label_len_one_short_df[Y_label_len_one_short_df['codes'] == code].copy()
    for name, code in dfs.items()
}

individual_dfs['QWAVE']

,ecg_id,patient_id,strat_fold,label,codes,len
259,267,7662.0,1,"[QWAVE, SR]",QWAVE,1
302,310,1316.0,6,"[QWAVE, SR]",QWAVE,1
315,323,3825.0,2,"[QWAVE, SR]",QWAVE,1
579,592,2885.0,3,"[QWAVE, SR]",QWAVE,1
587,600,4590.0,5,"[QWAVE, SR]",QWAVE,1
...,...,...,...,...,...,...
18530,18569,10055.0,6,"[QWAVE, SR]",QWAVE,1
18846,18885,12730.0,3,"[QWAVE, SR]",QWAVE,1
19300,19339,13881.0,2,"[QWAVE, SR]",QWAVE,1
19332,19371,19680.0,2,"[QWAVE, SR]",QWAVE,1


In [65]:
#
# Cell 24: adds the dataframe "Y_all_other_pure" to the dictionary "individual_dfs"
#

individual_dfs['all_other']= Y_all_other_pure

individual_dfs['all_other']

,ecg_id,patient_id,strat_fold,label,codes,len
16,17,13619.0,9,"[ABQRS, AFIB, AFLT]",all_other,3
17,18,13619.0,9,[AFLT],all_other,1
19,20,13619.0,9,"[ABQRS, AFLT]",all_other,2
22,23,13619.0,9,[AFLT],all_other,1
25,26,13619.0,9,"[DIG, NST_, SR]",all_other,2
...,...,...,...,...,...,...
21779,21818,19204.0,10,"[ABQRS, SR, VCLVH]",all_other,2
21787,21826,9178.0,10,"[ABQRS, SARRH]",all_other,2
21788,21827,13862.0,5,"[ABQRS, SVARR]",all_other,2
21789,21828,13862.0,5,"[ABQRS, SR, PVC]",all_other,2


In [67]:
#
# Cell 24a: Check the fraction instances with of strat_fold == 10 
#

strat_fold_ten=(individual_dfs['QWAVE']['strat_fold']==10).value_counts()
strat_fold_ten

strat_fold
False    202
True      22
Name: count, dtype: int64

In [69]:
#
# Cell 25: Selects from the dataframe in individual_dfs all the instances
# with strat_fold == 10 an put them into separate dataframe
#

test_dfs = {name: df[df['strat_fold'] == 10].copy() for name, df in individual_dfs.items()}
test_dfs['QWAVE']

,ecg_id,patient_id,strat_fold,label,codes,len
683,696,7153.0,10,"[QWAVE, SR]",QWAVE,1
1604,1617,5003.0,10,"[QWAVE, SR]",QWAVE,1
1937,1950,3267.0,10,"[QWAVE, SR]",QWAVE,1
2761,2776,11700.0,10,"[QWAVE, SR]",QWAVE,1
3093,3108,6121.0,10,"[QWAVE, SR]",QWAVE,1
3614,3629,9001.0,10,"[QWAVE, SR]",QWAVE,1
6329,6350,7308.0,10,"[QWAVE, SR]",QWAVE,1
6755,6776,11613.0,10,"[QWAVE, SR]",QWAVE,1
6773,6794,11613.0,10,"[QWAVE, SR]",QWAVE,1
6941,6962,5910.0,10,"[QWAVE, SR]",QWAVE,1


In [71]:
#
# Cell 25: substracts for all classes the corresponding test instances from the sets.
# Generates the dictionary "residue_dfs" with all the residue df of the individual classes  
#

residue_dfs = {name: df[~df.index.isin(test_dfs[name].index)].copy() for name, df in individual_dfs.items()}

residue_dfs['QWAVE']

,ecg_id,patient_id,strat_fold,label,codes,len
259,267,7662.0,1,"[QWAVE, SR]",QWAVE,1
302,310,1316.0,6,"[QWAVE, SR]",QWAVE,1
315,323,3825.0,2,"[QWAVE, SR]",QWAVE,1
579,592,2885.0,3,"[QWAVE, SR]",QWAVE,1
587,600,4590.0,5,"[QWAVE, SR]",QWAVE,1
...,...,...,...,...,...,...
18530,18569,10055.0,6,"[QWAVE, SR]",QWAVE,1
18846,18885,12730.0,3,"[QWAVE, SR]",QWAVE,1
19300,19339,13881.0,2,"[QWAVE, SR]",QWAVE,1
19332,19371,19680.0,2,"[QWAVE, SR]",QWAVE,1


In [97]:
#
# Cell 26: splits the df in the dictionary "residue_dfs" in training and validation sets.
#

training_dfs = {
    name: df.sample(frac=0.75, random_state=7).copy()
    for name, df in residue_dfs.items()
}


validation_dfs = {
    name.replace('_training', '_validation'): df.drop(training_dfs[name].index).copy()
    for name, df in residue_dfs.items()
}


display(training_dfs['QWAVE'][0:5]) 
display(validation_dfs['QWAVE'][0:5])

,ecg_id,patient_id,strat_fold,label,codes,len
5998,6019,4100.0,2,"[QWAVE, SR]",QWAVE,1
8953,8977,7062.0,9,"[QWAVE, SR]",QWAVE,1
1477,1490,20956.0,9,"[QWAVE, SR]",QWAVE,1
687,700,7361.0,7,"[QWAVE, SR]",QWAVE,1
16037,16075,2648.0,6,"[QWAVE, SR]",QWAVE,1


,ecg_id,patient_id,strat_fold,label,codes,len
259,267,7662.0,1,"[QWAVE, SR]",QWAVE,1
302,310,1316.0,6,"[QWAVE, SR]",QWAVE,1
587,600,4590.0,5,"[QWAVE, SR]",QWAVE,1
601,614,938.0,4,"[QWAVE, SR]",QWAVE,1
676,689,3805.0,6,"[QWAVE, SR]",QWAVE,1


In [99]:
#
# Cell 27: count the number of instances splits the df in the dictionary "residue_dfs" in training and validation sets.
#

dfs_all= {1: test_dfs,
          2: training_dfs,
          3: validation_dfs
         }

summation=sum(df.shape[0] for df in dfs_all.values() for df in df.values())
print('Total sum of all instances in the different sets: ',summation)

Total sum of all instances in the different sets:  21297


In [101]:
#
# Cell 28: combine all test / validation / training sets of all sets (all classes built)
#

train_df = pd.concat(training_dfs.values(), axis=0).sample(frac=1, random_state=7)
val_df   = pd.concat(validation_dfs.values(), axis=0).sample(frac=1, random_state=7)
test_df  = pd.concat(test_dfs.values(), axis=0).sample(frac=1, random_state=7)
train_df

,ecg_id,patient_id,strat_fold,label,codes,len
11456,11483,10210.0,4,"[ABQRS, SR, VCLVH]",all_other,2
16411,16449,11369.0,8,"[ABQRS, STACH]",all_other,2
16084,16122,12765.0,2,[SR],SR,1
20016,20055,17424.0,1,"[ABQRS, SARRH]",all_other,2
14995,15032,10459.0,4,"[NDT, SR, PVC]",all_other,2
...,...,...,...,...,...,...
2980,2995,13993.0,4,[SR],SR,1
13732,13764,10203.0,7,[PACE],PACE,1
10254,10281,20883.0,3,[SR],SR,1
11935,11967,13805.0,7,[SARRH],SARRH,1


In [103]:
#
# Cell 29:
#

Y_final_dfs = {'train_df' : train_df, 'val_df': val_df, 'test_df' : test_df}

## Following cells (30 to 34) are for preparing X data without augmentation 

In [105]:
#
#Cell 30: loads original X values (no augmentation) 
#

from pathlib import Path

X_directory_path = Path("D:/ECG/X_zero")
X = np.load(X_directory_path/'X_float32.npy', mmap_mode='r')

In [107]:
#
# Cell 31: separets X values in training-, validation- and test data.
#

X_train = X[train_df.index.tolist(), :, :]
X_val   = X[val_df.index.tolist(), :, :]
X_test  = X[test_df.index.tolist(), :, :]

In [91]:
#
# Cell 32: creates a folder for data storage.
#

# Specify the directory name
directory_path = Path("D:/ECG/XY_diff_class_3")

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\XY_diff_class_3' already exists.


In [109]:
#
# Cell 33: stores the X data (numpy arrays) to archive in the created folder    
#

np.savez(directory_path/'X_np_arrays_3.npz', a = X_train, b = X_val, c = X_test)

In [111]:
#
# Cell 34: Saves Y_final_dfs and X_final_dfs
#

import pickle

with open(directory_path/'Y_data_3.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y_final_dfs, f) # serialize the list
    f.close()

#with open(directory_path/'X_data.pkl', 'wb') as f:  # open a text file
#    pickle.dump(X_final_dfs, f) # serialize the list
#    f.close()

## The following cells must be executed if one wants to use the augmented datasets of X

In [87]:
#
# Cell 35: path to read input data
#

X_directory_path = Path("D:/ECG/X_augmented_1000") # chose the right path beforhand
X = np.load(X_directory_path/'X_augmented.npz.npy', mmap_mode='r')

In [89]:
#
# Cell 36: divides X values in training-, validation- and test data.
#

X_train = X[train_df.index.tolist(), :, :]
X_val   = X[val_df.index.tolist(), :, :]
X_test  = X[test_df.index.tolist(), :, :]

In [91]:
#
# Cell 37: creates a folder for data storage.
#

from pathlib import Path

# Specify the directory name
directory_path = Path("D:/ECG/XY_diff_class_3_aug_1000") # chose the right path beforhand

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_path}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\XY_diff_class_3_aug_1000' created successfully.


In [93]:
#
# Cell 38: stores the X data (numpy arrays) to archive in the created folder    
#

np.savez(directory_path/'X_np_arrays.npz', a = X_train, b = X_val, c = X_test)

In [95]:
#
# Cell 39: Saves Y_final_dfs and X_final_dfs
#

import pickle

with open(directory_path/'Y_data.pkl', 'wb') as f:  # open a text file
    pickle.dump(Y_final_dfs, f) # serialize the list
    f.close()

#with open(directory_path/'X_data.pkl', 'wb') as f:  # open a text file
#    pickle.dump(X_final_dfs, f) # serialize the list
#    f.close()